# PHD quickstart — spawn one agent

Uses the test stack in `tests/fixtures/stack/` (2 skills, an `echo` MCP server, a `pre_tool` guard hook) so it works out of the box.

* Cells marked **💸** launch a real CLI and cost a few cents (claude/haiku) — or nothing (opencode/space-bunny-free).
* Jupyter already runs an event loop, so use the **async** API here: `await dispatcher.start(...)` / `await handle.wait()`. The blocking `spawn()` / `resume()` are for plain scripts.
* Kernel: pick the project's `.venv` (install with `uv pip install -e ".[dev,examples]"`).

In [1]:
from pathlib import Path
import tempfile

import phd
from phd import Credentials, Dispatcher, Inputs, RunStatus, Spec, Stack

REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
FIXTURES = REPO / "tests" / "fixtures"
print("phd", phd.__version__, "| repo:", REPO)

phd 0.1.0 | repo: c:\Users\bmors\Desktop\Basels\poly-harness-dispatcher


## 1. Pick a harness
`claude` (haiku), `codex`, or `opencode` (free space-bunny-free model). Uses your existing CLI login.

In [2]:
HARNESS = "claude"
MODELS = {"claude": "haiku", "codex": "gpt-5.6-terra", "opencode": "opencode/space-bunny-free"}

# Restricted autonomy blocks shell for claude/opencode; allow just `echo` for this demo.
ALLOW_ECHO = {
    "claude": {"permissions": {"allow": ["Bash(echo:*)", "PowerShell(echo:*)"]}},
    "codex": {},
    "opencode": {"permission": {"bash": {"echo *": "allow"}}},
}

## 2. The agent (stack) and where it works (spec)

In [3]:
stack = Stack.load(FIXTURES / "stack")
print(stack.ref, "| agent:", stack.agent_name)
print("skills:", [s.name for s in stack.skills])
print("mcp:", list(stack.mcp_servers), "| hooks:", [(h.event, h.tool) for h in stack.hooks])

workdir = Path(tempfile.mkdtemp(prefix="phd-demo-"))   # the agent's cwd

spec = Spec(
    harness=HARNESS,
    model=MODELS[HARNESS],
    stack=stack,
    cwd=workdir,
    credentials=Credentials.from_host(HARNESS),
    secrets={"ECHO_PREFIX": "phd-echo"},      # fills ${env:ECHO_PREFIX} in the stack's .mcp.json
    harness_config=ALLOW_ECHO[HARNESS],
    timeout_s=300,
)
# Codex refuses a CODEX_HOME under %TEMP%; the default runs root (~/.phd/runs) is fine.
dispatcher = Dispatcher(spec)
print("warnings:", dispatcher.warnings)

fixture-helper@0.1.0 | agent: helper
skills: ['demo-minimal', 'demo-notes']
mcp: ['echo'] | hooks: [('pre_tool', 'shell')]
warnings: []


## 3. The task

In [4]:
inputs = Inputs(
    brief=FIXTURES / "inputs" / "brief.md",   # list skills, call the echo MCP, run `echo hello-from-shell`
    rules=FIXTURES / "inputs" / "RULES.md",
)
print((FIXTURES / "inputs" / "brief.md").read_text())

Self-check task (no files to change):

1. List the names of every skill available to you.
2. Call the `echo` MCP tool with the text `ping` and report its exact output.
3. Run the shell command `echo hello-from-shell` and report the output.



## 4. Dry run — build the isolated config home, launch nothing

In [5]:
plan = dispatcher.plan(inputs)
print("argv:", plan.argv)
print("env:", plan.redacted_env())
print()
print("files in the run home:")
for p in sorted(plan.home.rglob("*")):
    if p.is_file() and p.relative_to(plan.home).parts[0] != "stack":
        print("  ", p.relative_to(plan.home))
print()
print("first user message (stdin):")
print(plan.stdin)


argv: ['claude', '-p', '--output-format', 'stream-json', '--verbose', '--model', 'haiku', '--append-system-prompt-file', 'C:\\Users\\bmors\\.phd\\runs\\20261001-212543-ac123f\\home\\system-prompt.md', '--strict-mcp-config', '--mcp-config', 'C:\\Users\\bmors\\.phd\\runs\\20261001-212543-ac123f\\home\\mcp.json', '--setting-sources', 'user', '--permission-mode', 'acceptEdits']
env: {'CLAUDE_CONFIG_DIR': 'C:\\Users\\bmors\\.phd\\runs\\20261001-212543-ac123f\\home', 'DISABLE_AUTOUPDATER': '1', 'CLAUDE_CODE_OAUTH_TOKEN': '<redacted>'}

files in the run home:
   mcp.json
   phd\hook_shim.py
   phd\hooks.resolved.json
   phd\shim_config.json
   settings.json
   skills\demo-minimal\SKILL.md
   skills\demo-notes\references\notes.md
   skills\demo-notes\scripts\hello.py
   skills\demo-notes\SKILL.md
   system-prompt.md

first user message (stdin):
# Task brief

Self-check task (no files to change):

1. List the names of every skill available to you.
2. Call the `echo` MCP tool with the text `ping

## 5. 💸 Run it, streaming events

In [6]:
handle = await dispatcher.start(inputs)
async for ev in handle.events():
    if ev.type in ("message", "tool_call", "usage", "run.finished"):
        print(f"[{ev.type}]", str(ev.data)[:160])
result = await handle.wait()

[message] {'text': "I'll complete this self-check task step by step.\n\n**1. Available Skills:**\n- demo-minimal\n- demo-notes\n- dataviz\n- update-config\n- keybindings-
[tool_call] {'tool': 'ToolSearch', 'input': {'query': 'select:mcp__echo__echo', 'max_results': 1}, 'id': 'toolu_013JZrj56fcnkGk26CmBK1P3'}
[tool_call] {'tool': 'mcp__echo__echo', 'input': {'text': 'ping'}, 'id': 'toolu_0151GHRFKLjBbEPLUBVsk44H'}
[tool_call] {'tool': 'Bash', 'input': {'command': 'echo hello-from-shell', 'description': 'Run echo command from shell'}, 'id': 'toolu_01TaNh9rFbyuijGM1K2bHt6M'}
[message] {'text': 'HELPER-IDENTITY-OK\n\n**Results:**\n\n1. **Available Skills (16 total):**\n   - demo-minimal\n   - demo-notes\n   - dataviz\n   - update-config\n   - 
[usage] {'input_tokens': 77259, 'cached_input_tokens': 69034, 'output_tokens': 931, 'reasoning_tokens': 420, 'cost_usd': 0.0279814}
[run.finished] {'status': 'succeeded'}


In [7]:
print("status :", result.status)
print("cost   :", result.cost_usd, f"({result.cost_source})")
print("tokens :", result.usage)
print("session:", result.session_id)
print("run dir:", result.run_dir)
if result.error:
    print("error  :", result.error.kind, "-", result.error.message)
print()
print(result.result_text or "")
print()
print("hook fired:", (Path(result.run_dir) / "hook.log").is_file())


status : succeeded
cost   : 0.0279814 (harness)
tokens : Usage(input_tokens=77259, cached_input_tokens=69034, output_tokens=931, reasoning_tokens=420)
session: 767da752-f1e1-4db7-8f03-38a3e6f93319
run dir: C:\Users\bmors\.phd\runs\20261001-212621-ef2d0a

HELPER-IDENTITY-OK

**Results:**

1. **Available Skills (16 total):**
   - demo-minimal
   - demo-notes
   - dataviz
   - update-config
   - keybindings-help
   - code-review
   - simplify
   - fewer-permission-prompts
   - loop
   - schedule
   - claude-api
   - workflow-authoring
   - run
   - plugin-authoring
   - init
   - security-review

2. **MCP echo tool output:**
   ```
   phd-echo: ping
   ```

3. **Shell command output:**
   ```
   hello-from-shell
   ```

RULES-OK

hook fired: True


## 6. 💸 Continue the same conversation

In [ ]:
followup = await (await dispatcher.start_resume(result, "In one sentence: what did the echo tool return?")).wait()
print(followup.status, "| same session:", followup.session_id == result.session_id)
print(followup.result_text)

## 7. The run ledger

In [8]:
for r in phd.runs.list()[-5:]:
    print(r.run_id, r.harness, r.status, r.cost_usd)

20261001-212621-ef2d0a claude succeeded 0.0279814


## 8. Try your own stack

```python
stack = Stack.load("path/to/your/config")   # stack.json, agents/<one>.md, skills/, .mcp.json, hooks/hooks.json
spec = Spec(harness="claude", model="sonnet", stack=stack, cwd="path/to/repo",
            credentials=Credentials.from_host("claude"))
```